In [2]:
# INSTALLAZIONE DIPENDENZE 
!pip install -q -U bitsandbytes
!pip install -q -U accelerate
!pip install -q -U datasets
!pip install -q -U git+https://github.com/huggingface/transformers.git

print("Installazione completata.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 7.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 10.1 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 10.6 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0.dev0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
Installazione completata.


In [ ]:
import pandas as pd
import json
import torch
import bitsandbytes
import os
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

2026-01-19 18:08:48.410949: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768846128.644770      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768846128.718610      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768846129.271863      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768846129.271901      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768846129.271904      55 computation_placer.cc:177] computation placer alr

In [ ]:

# ==========================================
# 1. CONFIGURAZIONE
# ==========================================

INPUT_FILE = "/kaggle/input/jokes-sample/jokes_sample.csv" 
OUTPUT_FILE = "dpo_dataset_ready.jsonl"

# MODELLO TEACHER: Usiamo Zephyr (o Mistral) perché segue bene le istruzioni
# Zephyr è un Mistral finetunato per essere un assistente, perfetto per questo task.
MODEL_ID = "HuggingFaceH4/zephyr-7b-beta" 

# ==========================================
# 2. CARICAMENTO DATASET FILTRATO
# ==========================================
print(f"Carico il dataset filtrato: {INPUT_FILE}...")

if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    df_subset = df.head(2000).copy()
    # Assicuriamoci che la colonna testo si chiami 'jokeText'
    text_col = "jokeText" if "jokeText" in df_subset.columns else "text"
    jokes = df_subset[text_col].tolist()
    print(f"Caricate {len(jokes)} battute Gold.")
else:
    raise FileNotFoundError(f"Non trovo {INPUT_FILE}.")



# ==========================================
# 3. CARICAMENTO MODELLO TEACHER
# ==========================================
print(f"Carico il modello Teacher: {MODEL_ID}...")

# Usiamo pipeline per semplicità. 
# torch_dtype=torch.bfloat16 risparmia memoria sulla GPU
pipe = pipeline(
    "text-generation", 
    model=MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

# ==========================================
#  4. GENERAZIONE COPPIE (Il Cuore)
# ==========================================
dpo_data = []

# Prompt di sistema per forzare il modello a essere noioso
system_prompt = "You are a literal, boring assistant. You ruin jokes by explaining them literally or rewriting them as serious statements."

print("Inizio generazione versioni noiose (Rejected)...")

for joke in tqdm(jokes):
    # Costruiamo la chat
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Rewrite the following joke into a completely serious, boring, and literal sentence. Remove all humor. Keep it short.\n\nJoke: \"{joke}\""}
    ]
    
    try:
        # Generiamo
        outputs = pipe(
        messages, 
        max_new_tokens=60,   # Genera al massimo 60 "parole" (token)
        do_sample=True,      # Non essere ripetitivo (non usare Greedy Search)
        temperature=0.7,     # Creatività media
        top_k=50,            # Taglia le parole improbabili
        top_p=0.95           # Nucleus Sampling
        )
        
        # Estraiamo il testo (Zephyr/Mistral restituiscono un dizionario)
        generated = outputs[0]["generated_text"][-1]["content"]
        
        # Pulizia: a volte il modello ripete l'input o mette virgolette
        boring_text = generated.replace('"', '').replace("Here is a serious version:", "").strip()

        # Creiamo la riga per DPO
        entry = {
            "prompt": "Tell me a joke.",  # Prompt generico che useremo nel training
            "chosen": joke,               # La tua battuta Gold
            "rejected": boring_text       # La versione rovinata
        }
        
        dpo_data.append(entry)
        
    except Exception as e:
        print(f" Errore su battuta: {e}")
        continue

# ==========================================
# 5. SALVATAGGIO JSONL
# ==========================================
with open(OUTPUT_FILE, 'w') as f:
    for entry in dpo_data:
        json.dump(entry, f)
        f.write('\n')

print(f"\n FATTO! Dataset salvato: {OUTPUT_FILE}")
print(f"   Contiene {len(dpo_data)} coppie pronte per il DPOTrainer.")


Carico il dataset filtrato: /kaggle/input/jokes-sample/jokes_sample.csv...
Caricate 2000 battute Gold.
Carico il modello Teacher: HuggingFaceH4/zephyr-7b-beta...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Device set to use cuda:0


Inizio generazione versioni noiose (Rejected)...


100%|██████████| 2000/2000 [2:03:55<00:00,  3.72s/it]  


 FATTO! Dataset salvato: dpo_dataset_ready.jsonl
   Contiene 2000 coppie pronte per il DPOTrainer.
